# Dedekind DSL: Analyst Tier
Declarative analyst pipeline using combinators instead of pandas call-chain spaghetti.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

try:
    from dedekind import SetDef, table
except ModuleNotFoundError:
    candidate_roots = [Path.cwd(), *Path.cwd().parents]
    for root in candidate_roots:
        candidate = root / "python"
        if (candidate / "dedekind").exists():
            sys.path.insert(0, str(candidate))
            break
    from dedekind import SetDef, table

print("Imported Analyst combinators: table, join, group_by/summarize, pivot/unpivot")

ModuleNotFoundError: No module named 'dedekind'

## Part 1: Source Tables
Define compact source tables and wrap them with the `table(...)` combinator.

In [ ]:
df_sales = pd.DataFrame(
    [
        {"date": "2026-01-05", "product_id": "p1", "region": "north", "units": 10, "revenue": 100},
        {"date": "2026-01-07", "product_id": "p2", "region": "north", "units": 5, "revenue": 250},
        {"date": "2026-01-09", "product_id": "p3", "region": "south", "units": 7, "revenue": 140},
        {"date": "2026-02-03", "product_id": "p1", "region": "south", "units": 8, "revenue": 80},
        {"date": "2026-02-10", "product_id": "p2", "region": "north", "units": 4, "revenue": 200},
        {"date": "2026-02-11", "product_id": "p3", "region": "south", "units": 10, "revenue": 200},
    ]
)

df_products = pd.DataFrame(
    [
        {"product_id": "p1", "category": "Hardware"},
        {"product_id": "p2", "category": "Software"},
        {"product_id": "p3", "category": "Accessories"},
    ]
)

df_regions = pd.DataFrame(
    [
        {"region": "north", "segment": "Enterprise"},
        {"region": "south", "segment": "SMB"},
    ]
)

sales = table("sales", df_sales)
products = table("products", df_products)
regions = table("regions", df_regions)

display(df_sales.head())

## Part 2: Declarative Pipeline
Compose the report with combinators (`join`, `derive`, `group_by`, `summarize`, `order_by`).

In [ ]:
report_plan = (
    sales
    .join(products, on="product_id", cardinality="many_to_one")
    .join(regions, on="region", cardinality="many_to_one")
    .derive(month=lambda t: pd.to_datetime(t["date"]).dt.to_period("M").astype(str))
    .group_by(["month", "category"])
    .summarize(revenue=("revenue", "sum"), units=("units", "sum"))
    .order_by(["month", "category"])
)

summary = report_plan.realize()

print("logical plan")
print(report_plan.explain())

print("\nsummary")
display(summary)

## Part 3: Pivot, Unpivot, And Contracts
Use reshape combinators and simple guardrails; core relational pivot remains tracked in issue #170.

In [ ]:
safe_join = sales.join(products, on="product_id", cardinality="many_to_one")
safe_join.expect_no_multiplicity_increase()

pivot_plan = (
    report_plan
    .pivot(index="month", columns="category", values="revenue", aggfunc="sum", fill_value=0)
    .order_by("month")
)
pivoted = pivot_plan.realize()

unpivoted = (
    pivot_plan
    .unpivot(id_vars=["month"], var_name="category", value_name="revenue")
    .order_by(["month", "category"])
    .realize()
)

region_values = regions.to_set("region").realize()

print("pivoted revenue matrix")
display(pivoted)

print("unpivoted back to long form")
display(unpivoted)

print("realized region values:", region_values)

In [ ]:
pivot_check = pivoted.sort_values("month").reset_index(drop=True)

expected_columns = ["month", "Accessories", "Hardware", "Software"]
assert list(pivot_check.columns) == expected_columns, pivot_check.columns.tolist()

expected_rows = [
    {"month": "2026-01", "Accessories": 140, "Hardware": 100, "Software": 250},
    {"month": "2026-02", "Accessories": 200, "Hardware": 80, "Software": 200},
]
assert pivot_check.to_dict(orient="records") == expected_rows

assert sorted(region_values) == ["north", "south"]

plan_text = report_plan.explain()
assert "join" in plan_text and "summarize" in plan_text

print("analyst combinator flow verified")
print("notebook-03-ok")